In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ================================================================
# codellama_vuln_optimized.py
# CodeLlama-7b-Instruct — PrimeVul vulnerability classification
#
# Key improvements over baseline:
#   1. Unbiased YES/NO prompt (no label words in prompt body)
#   2. Correct CodeLlama [INST] / <<SYS>> template
#   3. Code preprocessing: strip comments, includes, macros
#   4. bfloat16 precision (safer than float16 on Kaggle T4/P100)
#   5. Logit-level scoring — reads token probabilities instead of
#      free text, giving a calibrated confidence score per sample
#   6. Majority-vote fallback when logit score is ambiguous
#   7. MCC + confusion matrix + skip-rate diagnostics
#   8. Incremental CSV saves so a crash loses at most BATCH_SAVE rows
# ================================================================

# !pip install transformers accelerate scikit-learn tqdm pandas -q

import torch
import re
import json
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
)

# ================================================================
# CONFIG — change these without touching anything else
# ================================================================
MODEL_NAME     = "codellama/CodeLlama-7b-Instruct-hf"
MAX_CODE_TOKS  = 512       # tokens reserved for code; rest is prompt overhead
MAX_NEW_TOKS   = 20        # CodeLlama-7B is more verbose; needs a bit more room
VOTE_ROUNDS    = 3         # odd number → no tie; round 0 = greedy, rest sampled
VOTE_TEMP      = 0.6       # temperature for rounds 1+
LOGIT_MODE     = True      # if True, score via token logits (most accurate)
LOGIT_THRESH   = 0.0       # log-odds threshold: >0 → YES, <0 → NO
N_SAMPLES      = None      # None = full dataset; int = debug subset
BATCH_SAVE     = 50        # flush per-sample log every N rows
OUT_DIR        = "/kaggle/working"

# ================================================================
# AUTO-DETECT DATASET
# ================================================================
file_path = None
for root, _, files in os.walk("/kaggle/input"):
    for fname in files:
        if "primevul_test_paired" in fname:
            file_path = os.path.join(root, fname)
            break
    if file_path:
        break

if file_path is None:
    raise FileNotFoundError("primevul_test_paired not found in /kaggle/input")
print(f"✅ Dataset : {file_path}")

# ================================================================
# LOAD MODEL
# bfloat16: same memory as float16, far fewer NaN/Inf events on
# T4/A100. device_map="auto" handles multi-GPU or CPU offload.
# ================================================================
print(f"⏳ Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# CodeLlama tokenizer sometimes ships without a pad token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("✅ Model loaded\n")

# ================================================================
# PRE-CACHE YES / NO TOKEN IDs
# We score the very first generated token's log-probability for
# "Yes" vs "No". Pre-caching avoids repeated tokenizer calls.
# CodeLlama BPE: "Yes"→29891, "No"→29871 — but we look them up
# dynamically so this works even if the vocab changes.
# ================================================================
YES_IDS = tokenizer.encode("Yes", add_special_tokens=False)  # usually [29979, 267] or similar
NO_IDS  = tokenizer.encode("No",  add_special_tokens=False)

# Use only the first sub-token of each word for scoring
YES_TOK = YES_IDS[0]
NO_TOK  = NO_IDS[0]
print(f"   YES token id: {YES_TOK}  ({tokenizer.decode([YES_TOK])})")
print(f"   NO  token id: {NO_TOK}   ({tokenizer.decode([NO_TOK])})\n")

# ================================================================
# CODE PREPROCESSOR
# PrimeVul functions often begin with hundreds of tokens of
# #include chains, #ifdef guards, and doc-comment blocks.
# Strip these so the actual logic fits in MAX_CODE_TOKS.
# ================================================================
_ML_COMMENT = re.compile(r'/\*.*?\*/', re.DOTALL)
_SL_COMMENT = re.compile(r'//[^\n]*')
_DIRECTIVE  = re.compile(r'^\s*#[^\n]*', re.MULTILINE)
_BLANK3     = re.compile(r'\n{3,}')

def preprocess(code: str) -> str:
    code = _ML_COMMENT.sub('', code)
    code = _SL_COMMENT.sub('', code)
    code = _DIRECTIVE.sub('', code)
    code = _BLANK3.sub('\n\n', code)
    return code.strip()

# ================================================================
# PROMPT BUILDER — CodeLlama [INST] / <<SYS>> format
#
# CRITICAL DESIGN CHOICE:
# Do NOT put "SAFE" or "VULNERABLE" anywhere in the prompt.
# CodeLlama-7B is strong enough to follow abstract instructions,
# but all LLMs are biased toward completing with words that
# appear in their immediate context. Using YES/NO eliminates
# this bias entirely while remaining unambiguous.
#
# The <<SYS>> block sets the persona; [INST]...[/INST] frames
# the user turn. CodeLlama-Instruct was fine-tuned on this exact
# template — deviating from it degrades instruction-following.
# ================================================================
SYS_PROMPT = (
    "You are an expert C/C++ security auditor with deep knowledge "
    "of memory safety, integer overflow, format string bugs, and "
    "injection vulnerabilities. You answer only with Yes or No."
)

def build_prompt(code: str) -> str:
    code = preprocess(code)

    # Truncate at token level
    toks = tokenizer.encode(code, add_special_tokens=False)
    if len(toks) > MAX_CODE_TOKS:
        code = tokenizer.decode(toks[:MAX_CODE_TOKS], skip_special_tokens=True)
        code += "\n// [truncated]"

    # Official CodeLlama-Instruct chat template
    return (
        f"<s>[INST] <<SYS>>\n{SYS_PROMPT}\n<</SYS>>\n\n"
        f"Does the following C/C++ function contain a security vulnerability "
        f"(buffer overflow, use-after-free, integer overflow, format string bug, "
        f"SQL/command injection, or similar CWE)?\n\n"
        f"```c\n{code}\n```\n\n"
        f"Answer with exactly one word — Yes or No. [/INST]"
    )

# ================================================================
# LABEL EXTRACTOR (text-based fallback)
# Only used when LOGIT_MODE=False or logit scoring fails.
# Scans only the newly generated tokens; stops at first match.
# ================================================================
_YES_RE = re.compile(r'\byes\b', re.IGNORECASE)
_NO_RE  = re.compile(r'\bno\b',  re.IGNORECASE)

def extract_yn(raw: str) -> int:
    """1=vulnerable, 0=safe, -1=unparseable."""
    for line in raw.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        y = _YES_RE.search(line)
        n = _NO_RE.search(line)
        if y and n:
            return 1 if y.start() < n.start() else 0
        if y:
            return 1
        if n:
            return 0
    return -1

# ================================================================
# CORE GENERATION HELPERS
# ================================================================
def _tokenize(prompt: str):
    return tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
        padding=False,
    ).to(model.device)

def _generate_text(inputs, temperature: float = 0.0) -> str:
    """Generate free text and return the new tokens decoded."""
    kwargs = dict(
        max_new_tokens=MAX_NEW_TOKS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if temperature == 0.0:
        kwargs["do_sample"] = False
    else:
        kwargs["do_sample"] = True
        kwargs["temperature"] = temperature
        kwargs["top_p"] = 0.92

    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)

    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True)

def _score_logits(inputs) -> float | None:
    """
    Return log-odds score: log P(YES) - log P(NO) at position 0.
    Positive  → model favours YES (vulnerable).
    Negative  → model favours NO  (safe).
    None      → forward pass failed.

    This is strictly better than reading free text because:
      - It uses the full probability distribution, not just argmax.
      - It gives a calibrated confidence score you can threshold.
      - It is deterministic (no sampling variance).
    """
    try:
        with torch.no_grad():
            logits = model(**inputs).logits  # (1, seq_len, vocab)
        # logits[:, -1, :] = distribution over the NEXT token (first generated)
        next_logits = logits[0, -1, :]
        log_probs   = torch.log_softmax(next_logits, dim=-1)
        score = (log_probs[YES_TOK] - log_probs[NO_TOK]).item()
        return score
    except Exception as e:
        print(f"   [logit score failed: {e}]")
        return None

# ================================================================
# MAIN PREDICT FUNCTION
# Strategy:
#   1. If LOGIT_MODE: score via token log-odds (best accuracy).
#      If logit scoring fails, fall back to majority vote.
#   2. Majority vote: greedy decode + (VOTE_ROUNDS-1) sampled
#      decodes. Final = majority of valid votes.
# ================================================================
def predict(code: str) -> tuple[int, dict]:
    """
    Returns (label, metadata_dict).
    label: 1=vulnerable, 0=safe, -1=skip
    """
    prompt  = build_prompt(code)
    inputs  = _tokenize(prompt)
    meta    = {"mode": "?", "score": None, "votes": [], "raw": ""}

    # ── LOGIT MODE (preferred) ──────────────────────────────────
    if LOGIT_MODE:
        score = _score_logits(inputs)
        if score is not None:
            meta["mode"]  = "logit"
            meta["score"] = round(score, 4)
            label = 1 if score > LOGIT_THRESH else 0
            return label, meta

    # ── MAJORITY VOTE FALLBACK ──────────────────────────────────
    votes = []
    last_raw = ""
    for i in range(VOTE_ROUNDS):
        temp    = 0.0 if i == 0 else VOTE_TEMP
        raw     = _generate_text(inputs, temperature=temp)
        last_raw = raw
        v = extract_yn(raw)
        if v != -1:
            votes.append(v)

    meta["mode"] = "vote"
    meta["votes"] = votes
    meta["raw"]   = last_raw[:120]

    if not votes:
        return -1, meta

    final = 1 if sum(votes) > len(votes) / 2 else 0
    return final, meta

# ================================================================
# LOAD DATASET
# ================================================================
dataset = []
with open(file_path) as f:
    for line in f:
        line = line.strip()
        if line:
            dataset.append(json.loads(line))

if N_SAMPLES:
    dataset = dataset[:N_SAMPLES]

print(f"✅ Loaded {len(dataset)} samples")

# ================================================================
# EVALUATION LOOP
# ================================================================
y_true, y_pred = [], []
skipped = 0
log_rows = []

print("\n🚀 Starting evaluation ...\n")

for i, sample in enumerate(tqdm(dataset)):
    code  = sample["func"]
    label = int(sample["target"])

    pred, meta = predict(code)

    log_rows.append({
        "index":      i,
        "true_label": label,
        "pred_label": pred,
        "mode":       meta["mode"],
        "score":      meta.get("score"),
        "votes":      str(meta.get("votes", [])),
        "raw":        meta.get("raw", "")[:100],
        "skipped":    pred == -1,
    })

    if pred == -1:
        skipped += 1
    else:
        y_true.append(label)
        y_pred.append(pred)

    # Incremental save — survive crashes on long runs
    if (i + 1) % BATCH_SAVE == 0:
        pd.DataFrame(log_rows).to_csv(f"{OUT_DIR}/codellama_per_sample.csv", index=False)

pd.DataFrame(log_rows).to_csv(f"{OUT_DIR}/codellama_per_sample.csv", index=False)

# ================================================================
# METRICS
# ================================================================
total = len(dataset)
print(f"\n{'='*55}")
print(f"  Total   : {total}")
print(f"  Valid   : {len(y_pred)}")
print(f"  Skipped : {skipped}  ({100*skipped/total:.1f}%)")
print(f"{'='*55}")

if not y_pred:
    print("\n❌ No valid predictions. Check prompt format & token IDs above.")
else:
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    cm   = confusion_matrix(y_true, y_pred)

    results = {
        "accuracy":  round(acc,  4),
        "precision": round(prec, 4),
        "recall":    round(rec,  4),
        "f1":        round(f1,   4),
        "mcc":       round(mcc,  4),   # PRIMARY metric for imbalanced binary
        "n_valid":   len(y_pred),
        "n_skipped": skipped,
    }

    print("\n🚀 FINAL RESULTS")
    print(f"   accuracy  : {acc:.4f}")
    print(f"   precision : {prec:.4f}")
    print(f"   recall    : {rec:.4f}")
    print(f"   f1        : {f1:.4f}")
    print(f"   mcc       : {mcc:.4f}   ← primary metric")

    print(f"\n📉 Confusion matrix (rows=true, cols=pred)")
    print(f"   {cm}")
    print(f"   TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")

    # Prediction balance diagnostic
    pred_pos = sum(y_pred) / len(y_pred)
    true_pos = sum(y_true) / len(y_true)
    print(f"\n   Pred positive rate : {pred_pos:.3f}")
    print(f"   True positive rate : {true_pos:.3f}")
    if abs(pred_pos - 0.5) < 0.06:
        print("   ⚠  Model still predicting ~50/50 — see diagnostics below")
    else:
        print("   ✅ Prediction distribution is skewed — model is discriminating")

    # If using logit mode, print score distribution for calibration insight
    if LOGIT_MODE:
        scores = [r["score"] for r in log_rows if r["score"] is not None]
        if scores:
            import statistics
            print(f"\n   Logit score stats:")
            print(f"   mean={statistics.mean(scores):.3f}  "
                  f"stdev={statistics.stdev(scores):.3f}  "
                  f"min={min(scores):.3f}  max={max(scores):.3f}")
            print(f"   (threshold={LOGIT_THRESH}; adjust if MCC is low but scores spread wide)")

    pd.DataFrame([results]).to_csv(f"{OUT_DIR}/codellama_primevul_results.csv", index=False)
    print(f"\n✅ Results saved → {OUT_DIR}/")

# ================================================================
# BUILT-IN DIAGNOSTICS
# Run these manually if MCC stays near 0 after the full run.
# ================================================================
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DIAGNOSTICS (run manually if MCC < 0.1)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Verify token IDs are correct:
   >>> print(tokenizer.decode([YES_TOK]), tokenizer.decode([NO_TOK]))
   Should print: Yes  No

2. Print 5 raw outputs (if vote mode):
   >>> for r in log_rows[:5]: print(r['raw'])

3. Check the rendered prompt for sample 0:
   >>> print(build_prompt(dataset[0]['func']))

4. If model always says the same thing:
   >>> scores = [r['score'] for r in log_rows if r['score']]
   >>> print(min(scores), max(scores))
   If range < 1.0, the model has no signal — try CodeLlama-13B.

5. Threshold tuning (if scores spread but MCC is low):
   >>> import numpy as np
   >>> thresholds = np.linspace(-3, 3, 61)
   >>> best_t, best_mcc = 0, -1
   >>> for t in thresholds:
   ...     preds = [1 if r['score'] > t else 0
   ...              for r in log_rows if r['score'] is not None]
   ...     trues = [r['true_label']
   ...              for r in log_rows if r['score'] is not None]
   ...     m = matthews_corrcoef(trues, preds)
   ...     if m > best_mcc: best_t, best_mcc = t, m
   >>> print(f"Best threshold: {best_t:.2f}  MCC: {best_mcc:.4f}")
   Then set LOGIT_THRESH = best_t and re-run.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

✅ Dataset : /kaggle/input/datasets/nikunjnawal009/primevul-codelalma/primevul_test_paired.jsonl
⏳ Loading codellama/CodeLlama-7b-Instruct-hf ...


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded

   YES token id: 3869  (Yes)
   NO  token id: 1939   (No)

✅ Loaded 870 samples

🚀 Starting evaluation ...




100%|██████████| 870/870 [50:22<00:00,  3.47s/it]


  Total   : 870
  Valid   : 870
  Skipped : 0  (0.0%)

🚀 FINAL RESULTS
   accuracy  : 0.4805
   precision : 0.4855
   recall    : 0.6552
   f1        : 0.5577
   mcc       : -0.0417   ← primary metric

📉 Confusion matrix (rows=true, cols=pred)
   [[133 302]
 [150 285]]
   TN=133  FP=302  FN=150  TP=285

   Pred positive rate : 0.675
   True positive rate : 0.500
   ✅ Prediction distribution is skewed — model is discriminating

   Logit score stats:
   mean=0.235  stdev=0.413  min=-1.062  max=1.188
   (threshold=0.0; adjust if MCC is low but scores spread wide)

✅ Results saved → /kaggle/working/

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DIAGNOSTICS (run manually if MCC < 0.1)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Verify token IDs are correct:
   >>> print(tokenizer.decode([YES_TOK]), tokenizer.decode([NO_TOK]))
   Should print: Yes  No

2. Print 5 raw outputs (if vote mode):
   >>> for r in log_rows[:5]: print(r['raw'])

3. Check the rendered prompt